In [8]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import numpy as np
from tensorflow.keras.models import load_model
import time
import os


#### Loading Face-Dectection+Classification Model

In [4]:
### Loading Face-Dectection+Classification Model
face_detection_model = load_model("face-detection.keras")

#### Loading Eye-Detection model

In [5]:

import keras
def rmse_loss(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred), axis=-1))


eye_detection_model = keras.models.load_model(
    "eye_detect_large_model1.keras",
    custom_objects={ "rmse_loss": rmse_loss},
)

#### EYES STATE DETECTION MODEL (OPEN/CLOSED)

In [7]:
closed_open =load_model("closedopen.keras")

In [13]:
import cv2
import numpy as np
import time
import os

# Create directory to save eye images if it doesn't exist
os.makedirs("eyes_image", exist_ok=True)

# cap = cv2.VideoCapture("drive2.mp4")
# cap = cv2.VideoCapture("myvideo2.mp4 ")
cap = cv2.VideoCapture(0)

frame_width = 672  # Desired width
frame_height = 672  # Desired height
cap.set(cv2.CAP_PROP_FRAME_WIDTH, frame_width)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, frame_height)

eye_image_counter = 0
last_saved_time = time.time()  # Track the time of the last save


left_eye_closed_counter = 0
right_eye_closed_counter = 0

alarm_triggered=False
alarm_counter=0

while True:
    ret, frame = cap.read()
    if not ret:  # when frame is not captured correctly
        break

    # Convert the frame to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized_for_face =cv2.resize(frame_rgb, (224, 224))


    img_resized = cv2.resize(frame_rgb, (224, 224)) / 255.0
    face_input=np.expand_dims(img_resized_for_face,axis=0)
    label, box = face_detection_model.predict(face_input ,verbose=0)
    prediction_of_face = label[0][0]
    x_min, y_min, x_max, y_max = box[0] * 224
    x_min, y_min, x_max, y_max = int(x_min), int(y_min), int(x_max), int(y_max)

    h, w, _ = frame.shape
    scale_x = w / 224
    scale_y = h / 224
    x_min = int(x_min * scale_x)
    x_max = int(x_max * scale_x)
    y_min = int(y_min * scale_y)
    y_max = int(y_max * scale_y)

    if prediction_of_face > 0.7:
        # 1. Showing rectangle
        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)
        cv2.putText(frame, f"DRIVER", (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)







        # Prepare the image for eye detection model
        eye_input = np.expand_dims(img_resized, axis=0)

        # Use the eye detection model to predict the eyes' bounding boxes
        eye_predictions = eye_detection_model.predict(eye_input ,verbose=0)
        lefteye_xcentre, lefteye_ycentre, lefteye_width, lefteye_height, righteye_xcentre, righteye_ycentre, righteye_width, righteye_height = eye_predictions[0] * 224

        # Rescale the eye coordinates from the 224x224 input image to the original frame size
        h, w, _ = frame.shape
        scale_x = w / 224
        scale_y = h / 224

        # Calculate bounding box for left eye
        lefteye_xmin = int((lefteye_xcentre - lefteye_width / 2) * scale_x)
        lefteye_ymin = int((lefteye_ycentre - lefteye_height / 2) * scale_y)
        lefteye_xmax = int((lefteye_xcentre + lefteye_width / 2) * scale_x)
        lefteye_ymax = int((lefteye_ycentre + lefteye_height / 2) * scale_y)

        # Calculate bounding box for right eye
        righteye_xmin = int((righteye_xcentre - righteye_width / 2) * scale_x)
        righteye_ymin = int((righteye_ycentre - righteye_height / 2) * scale_y)
        righteye_xmax = int((righteye_xcentre + righteye_width / 2) * scale_x)
        righteye_ymax = int((righteye_ycentre + righteye_height / 2) * scale_y)

        # Save images every 1 second
        left_eye_status="Awake"
        right_eye_status="Awake"
        left_eye_prob=0
        right_eye_prob=0

        if time.time() - last_saved_time >= 1:
            # Extract and save the left eye image
            left_eye_roi = frame[lefteye_ymin:lefteye_ymax, lefteye_xmin:lefteye_xmax]
            
            if left_eye_roi.size > 0:
                left_eye_resized = cv2.resize(left_eye_roi, (224, 224))
                input_for_model_lefteye = np.expand_dims(left_eye_resized ,axis=0)
                prediction_left = closed_open.predict(input_for_model_lefteye,verbose=0)
                left_eye_prob=prediction_left[0]
                if prediction_left[0]>0.8:
                    print(f"LEFT OPEN with{prediction_left[0]}")
                    left_eye_status="Awake"
                    left_eye_closed_counter = 0  #Resetting  counter if the eye is closed
                    
                elif prediction_left[0]<0.2:
                    print(f"LEFT closed with {prediction_left[0]}")
                    left_eye_status="Sleepy maybe"
                    left_eye_closed_counter+=1

                    if left_eye_closed_counter >= 2:
                        alarm_triggered = True

                left_eye_path = f"eyes_image/left_eye_{eye_image_counter}.jpg"
                cv2.imwrite(left_eye_path, left_eye_resized)

            # Extract and save the right eye 
            right_eye_roi = frame[righteye_ymin:righteye_ymax, righteye_xmin:righteye_xmax]
            if right_eye_roi.size > 0:
                right_eye_resized = cv2.resize(right_eye_roi, (224, 224))
                input_for_model_righteye =np.expand_dims(right_eye_resized ,axis=0)
                prediction_right = closed_open.predict(input_for_model_righteye,verbose=0)
                right_eye_prob=prediction_right[0]
                

                # print("Right eye prediction:",prediction_right)
                if prediction_right[0]>0.9:
                    print(f"RIGHT OPEN with{prediction_right[0]}")
                    right_eye_status="awake"
                    right_eye_closed_counter = 0

                elif prediction_right[0]<0.9:
                    print(f"Right closed with {prediction_right[0]}")
                    right_eye_status="sleepy maybe"
                    right_eye_closed_counter+=1
                    if right_eye_closed_counter>=2:
                        alarm_triggered=True
                    

                right_eye_path = f"eyes_image/right_eye_{eye_image_counter}.jpg"
                cv2.imwrite(right_eye_path, right_eye_resized)

           
            eye_image_counter += 1
            last_saved_time = time.time()

        # Draw rectangles around the detected eyes
        cv2.rectangle(frame, (lefteye_xmin, lefteye_ymin), (lefteye_xmax, lefteye_ymax), (0, 255, 0), 2)
        cv2.putText(frame, f"{left_eye_status}", (lefteye_xmin, lefteye_ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        cv2.rectangle(frame, (righteye_xmin, righteye_ymin), (righteye_xmax, righteye_ymax), (0, 255, 0), 2)
        cv2.putText(frame, f"{right_eye_status}", (righteye_xmin, righteye_ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)



        # ALARM DETECTED 
        if alarm_triggered:
            alarm_counter+=1
            cv2.putText(frame, "ALARM", (frame.shape[1]//3, frame.shape[0]//2), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
            alarm_triggered = False 


        # Display the alarm counter on the top-right corner
        cv2.putText(frame, f"Alarm Counter: {alarm_counter}", 
                    (frame.shape[1] - 200, 30),  # Top-right corner
                    cv2.FONT_HERSHEY_SIMPLEX, 
                    0.7, (0, 255, 255), 2, cv2.LINE_AA)







    # Display the frame with detected eyes
    cv2.imshow("Live Video", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


LEFT closed with [0.00170408]
RIGHT OPEN with[0.97227997]
LEFT closed with [0.01795751]
Right closed with [0.00410606]
LEFT closed with [0.10739119]
Right closed with [0.02086569]
Right closed with [0.73639244]
LEFT closed with [0.00936431]
Right closed with [0.00153433]
LEFT closed with [0.15795378]
Right closed with [0.00474954]
LEFT closed with [0.00112466]
RIGHT OPEN with[0.9996368]
Right closed with [0.18871954]
LEFT closed with [0.00628434]
RIGHT OPEN with[0.99094075]
LEFT closed with [0.03147168]
Right closed with [0.29321817]
